[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-4-llms-genai/03-structured-outputs-and-tool-calling/code/pydantic_basics.ipynb)

# Class 4.3 companion: Pydantic, properly

The main notebook used Pydantic to validate model output. This one is a focused tour of the library itself, because you will reuse it constantly: for tool schemas in Module 5 (agents) and for request and response models in FastAPI in Module 6. Everything here runs offline.

## Setup

```
pip install pydantic
```

## 1. A model is a typed contract

In [1]:
from pydantic import BaseModel

class Customer(BaseModel):
    name: str
    seats: int

# Build from keyword args...
c1 = Customer(name="Acme", seats=3)
print(c1.name, c1.seats, type(c1.seats).__name__)      # Acme 3 int

# ...and note Pydantic coerces compatible types: the string "5" becomes an int.
c2 = Customer(name="Beta", seats="5")
print(c2.seats, type(c2.seats).__name__)                # 5 int

Acme 3 int
5 int


## 2. Validation errors are specific and structured

In [2]:
from pydantic import ValidationError

try:
    c = Customer(name="Gamma", seats="a lot")     # not coercible to int
except ValidationError as e:
    err = e.errors()[0]
    print(err["loc"], "->", err["type"])       # ('seats',) -> int_parsing
else:
    print(c.name, c.seats)
# e.errors() is a list of dicts, so you can react programmatically, not just print.

('seats',) -> int_parsing


## 3. Field constraints: reject bad values, not just bad types

In [3]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

class Order(BaseModel):
    product: str
    quantity: int = Field(gt=0)                        # must be positive
    priority: Literal["low", "normal", "high"]         # must be one of these

print(Order(product="seats", quantity=3, priority="high"))

try:
    Order(product="seats", quantity=0, priority="high")   # violates gt=0
except ValidationError as e:
    print(e.errors()[0]["loc"], "->", e.errors()[0]["type"])   # ('quantity',) -> greater_than

product='seats' quantity=3 priority='high'
('quantity',) -> greater_than


## 4. Optional fields and defaults

In [4]:
from typing import Optional

class Ticket(BaseModel):
    summary: str
    assignee: Optional[str] = None       # may be missing
    urgent: bool = False                 # default when absent

t = Ticket(summary="cannot log in")
print(t.assignee, t.urgent)              # None False
print(Ticket(summary="x", urgent=True).urgent)   # True

None False
True


## 5. Nested models

In [5]:
class LineItem(BaseModel):
    sku: str
    qty: int

class Invoice(BaseModel):
    number: str
    items: list[LineItem]                # a list of sub-models

raw = '{"number": "INV-42", "items": [{"sku": "SEAT", "qty": 3}, {"sku": "ADDON", "qty": 1}]}'
inv = Invoice.model_validate_json(raw)
print(inv.items[0].sku, inv.items[0].qty)     # SEAT 3
print("total items:", sum(i.qty for i in inv.items))   # 4

SEAT 3
total items: 4


In [8]:
print(type(inv))

<class '__main__.Invoice'>


## 6. JSON in, JSON out

In [9]:
# Parse JSON text straight into a validated object...
o = Order.model_validate_json('{"product":"seats","quantity":2,"priority":"low"}')
# ...and serialize back to a dict or a JSON string.
print(o.model_dump())          # {'product': 'seats', 'quantity': 2, 'priority': 'low'}
print(o.model_dump_json())     # '{"product":"seats","quantity":2,"priority":"low"}'
# model_validate_json (parse) and model_dump_json (serialize) are the two you will
# use most when talking to an LLM or an API.

{'product': 'seats', 'quantity': 2, 'priority': 'low'}
{"product":"seats","quantity":2,"priority":"low"}


## Recap

Pydantic turns a class into a runtime contract: it coerces compatible types, rejects bad values with structured errors, supports constraints, defaults, and nesting, and converts to and from JSON in one call. That is why it is the backbone of validated LLM output here, tool schemas in Module 5, and API models in Module 6.